In [17]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import SequentialFeatureSelector

# Optional Explainability Libraries
try:
    import shap
except ImportError:
    shap = None  # SHAP is optional

try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    LimeTabularExplainer = None  # LIME is optional

# Data Loading and Preparation
def load_dataset(csv_path: str, max_rows=None):
    # Load CSV dataset
    df = pd.read_csv(csv_path)
    # If max_rows is set, sample a subset for faster processing
    if max_rows and df.shape[0] > max_rows:
        df = df.sample(n=max_rows, random_state=42).reset_index(drop=True)
    return df

def infer_target_column(df: pd.DataFrame):
    # Try to automatically detect target column based on common names
    possible_targets = ['target', 'power', 'generation', 'output', 'pv', 'y']
    for col in df.columns:
        if col.lower() in possible_targets:
            return col
    return df.columns[-1]  # fallback: use the last column

def prepare_features_and_target(df: pd.DataFrame, target_col: str):
    # Remove target column to get features
    features = df.drop(columns=[target_col])
    # Remove non-numeric columns (like datetime or strings)
    non_numeric = features.select_dtypes(exclude=[np.number]).columns
    features = features.drop(columns=non_numeric, errors='ignore')
    # Fill missing values with median for numeric columns
    features = features.fillna(features.median())

    X = features
    y = df[target_col]
    # Ensure target is numeric
    if not np.issubdtype(y.dtype, np.number):
        y = pd.to_numeric(y, errors='coerce').fillna(method='ffill')
    return X, y, X.columns.tolist()

# A1: Feature Correlation Heatmap
def correlation_heatmap(X: pd.DataFrame, save_path="correlation_heatmap.png"):
    # Compute correlation matrix
    corr = X.corr()
    # Plot heatmap
    plt.figure(figsize=(10, 8))
    plt.imshow(corr, cmap="coolwarm", interpolation='nearest', aspect='auto')
    plt.colorbar(label="Correlation Coefficient")
    # Set axis labels
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=8)
    plt.yticks(range(len(corr.columns)), corr.columns, fontsize=8)
    plt.title("Feature Correlation Heatmap")
    plt.tight_layout()
    # Save heatmap to file
    plt.savefig(save_path, dpi=200)
    plt.close()
    return corr, save_path

# Utility: Evaluate Regression Models
def evaluate_regressors(X_train, X_test, y_train, y_test, regressors: dict):
    # Train each model and compute RMSE, MAE, R2
    results = {}
    for name, model in regressors.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        results[name] = {
            "RMSE": mean_squared_error(y_test, preds) ** 0.5,
            "MAE": mean_absolute_error(y_test, preds),
            "R2": r2_score(y_test, preds)
        }
    return results

# A2/A3: PCA Feature Reduction & Model Evaluation
def apply_pca_and_evaluate(X, y, variance_threshold=0.99):
    # Standardize features before PCA
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Fit PCA to compute explained variance
    pca_full = PCA()
    pca_full.fit(X_scaled)
    cumsum = np.cumsum(pca_full.explained_variance_ratio_)
    # Determine number of components to retain given variance threshold
    n_components = np.searchsorted(cumsum, variance_threshold) + 1

    # Transform data with selected PCA components
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # Split data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)
    # Define regressors to evaluate
    regressors = {
        "RandomForest": RandomForestRegressor(n_estimators=30, random_state=42),
        "GradientBoosting": GradientBoostingRegressor(n_estimators=30, random_state=42),
        "Ridge": Ridge()
    }
    # Evaluate models
    results = evaluate_regressors(X_train, X_test, y_train, y_test, regressors)
    return n_components, results

# A4: Sequential Feature Selection
def sequential_feature_selection(X, y, n_features_to_select=6, direction='forward'):
    # Ensure requested features don't exceed total columns
    n_select = min(n_features_to_select, X.shape[1])
    # Use RandomForest as base estimator
    base_estimator = RandomForestRegressor(n_estimators=30, random_state=42)
    sfs = SequentialFeatureSelector(base_estimator,
                                    n_features_to_select=n_select,
                                    direction=direction)
    sfs.fit(X, y)

    # Get selected features
    selected_features = X.columns[sfs.get_support()]
    X_sel = X[selected_features]

    # Standardize selected features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_sel)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    # Evaluate models
    regressors = {
        "RandomForest": RandomForestRegressor(n_estimators=30, random_state=42),
        "GradientBoosting": GradientBoostingRegressor(n_estimators=30, random_state=42),
        "Ridge": Ridge()
    }
    results = evaluate_regressors(X_train, X_test, y_train, y_test, regressors)
    return list(selected_features), results

# A5: LIME Explainability
def explain_with_lime(model, X_train, X_test, feature_names):
    # Generate LIME explanation for a single instance
    if LimeTabularExplainer is None:
        return None, None
    explainer = LimeTabularExplainer(X_train, feature_names=feature_names, mode='regression')
    exp = explainer.explain_instance(X_test[0], model.predict, num_features=6)
    html_path = "lime_explanation.html"  # save file
    exp.save_to_file(html_path)
    return exp, html_path

# A5: SHAP Explainability
def explain_with_shap(model, X_train, feature_names, sample_size=5000):
    # Generate SHAP summary plot for a subset of data
    if shap is None:
        return None, None
    explainer = shap.TreeExplainer(model)
    sample_size = min(sample_size, X_train.shape[0])
    idx = np.random.choice(X_train.shape[0], size=sample_size, replace=False)
    X_sample = X_train[idx]
    shap_values = explainer.shap_values(X_sample)

    # Plot SHAP summary
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, features=X_sample, feature_names=feature_names, show=False)
    plt.tight_layout()
    save_path = "shap_summary.png"  # save figure
    plt.savefig(save_path, dpi=200)
    plt.close()
    return shap_values, save_path

# MAIN PIPELINE
def main(csv_path="/content/features_filtered.csv"):
    # Load dataset
    df = load_dataset(csv_path)
    target_col = infer_target_column(df)
    X, y, feature_names = prepare_features_and_target(df, target_col)

    print("Dataset shape:", X.shape)
    print("Target stats:", y.describe())

    # A1: Feature correlation
    corr, heatmap_path = correlation_heatmap(X)
    print(f"A1: Correlation heatmap saved at {heatmap_path}")

    # A2: PCA retaining 99% variance
    n99, res99 = apply_pca_and_evaluate(X, y, variance_threshold=0.99)
    print(f"\nA2: PCA retaining 99% variance → {n99} components")
    for name, metrics in res99.items():
        print(f"{name}: RMSE={metrics['RMSE']:.6f}, MAE={metrics['MAE']:.6f}, R2={metrics['R2']:.3f}")

    # A3: PCA retaining 95% variance
    n95, res95 = apply_pca_and_evaluate(X, y, variance_threshold=0.95)
    print(f"\nA3: PCA retaining 95% variance → {n95} components")
    for name, metrics in res95.items():
        print(f"{name}: RMSE={metrics['RMSE']:.6f}, MAE={metrics['MAE']:.6f}, R2={metrics['R2']:.3f}")

    # A4: Sequential feature selection
    forward_feats, forward_res = sequential_feature_selection(X, y, direction='forward')
    print(f"\nA4 (Forward SFS): Selected features = {forward_feats}")
    for name, metrics in forward_res.items():
        print(f"{name}: RMSE={metrics['RMSE']:.6f}, MAE={metrics['MAE']:.6f}, R2={metrics['R2']:.3f}")

    # Standardize full dataset for explainability
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    # Train RandomForest on full data
    model = RandomForestRegressor(n_estimators=30, random_state=42).fit(X_scaled, y)

    # A5: LIME explainability
    if LimeTabularExplainer:
        _, lime_path = explain_with_lime(model, X_scaled, X_scaled, feature_names)
        print(f"\nA5 (LIME): Explanation saved to {lime_path}")
    else:
        print("\nA5 (LIME): Package not available.")

    # A5: SHAP explainability
    if shap:
        _, shap_path = explain_with_shap(model, X_scaled, feature_names, sample_size=5000)
        print(f"A5 (SHAP): Summary plot saved to {shap_path}")
    else:
        print("A5 (SHAP): Package not available.")

if __name__ == "__main__":
    main()


Dataset shape: (39347, 257)
Target stats: count    3.934700e+04
mean     2.403463e-11
std      7.618344e-11
min      1.760298e-13
25%      3.369379e-12
50%      1.226787e-11
75%      2.698713e-11
max      3.498217e-09
Name: Tp8.__theta, dtype: float64
A1: Correlation heatmap saved at correlation_heatmap.png

A2: PCA retaining 99% variance → 104 components
RandomForest: RMSE=0.000000, MAE=0.000000, R2=-0.001
GradientBoosting: RMSE=0.000000, MAE=0.000000, R2=-0.001
Ridge: RMSE=0.000000, MAE=0.000000, R2=0.922

A3: PCA retaining 95% variance → 39 components
RandomForest: RMSE=0.000000, MAE=0.000000, R2=-0.001
GradientBoosting: RMSE=0.000000, MAE=0.000000, R2=-0.001
Ridge: RMSE=0.000000, MAE=0.000000, R2=0.878

A4 (Forward SFS): Selected features = ['run', 'onset_s', 'Af3.__alpha', 'Af3.__beta', 'Af3.__delta', 'Af3.__theta']
RandomForest: RMSE=0.000000, MAE=0.000000, R2=-0.001
GradientBoosting: RMSE=0.000000, MAE=0.000000, R2=-0.001
Ridge: RMSE=0.000000, MAE=0.000000, R2=0.078

A5 (LIME): 